# 12 — Ablation'lar (Adım 17), Kaggle

**Ayarlar:** Accelerator **GPU T4 x2** · Internet **On** · Secrets: `GH_TOKEN`. Sonra **Save Version → Save & Run All (Commit)**.

Yaklaşık **3,5 saat**. Kalori ablation'ları (~1 saat) önce biter ve **ara kayıt** olarak GitHub'a gider; ViT 384 px sınıflandırma (~2,5 saat) en sonda.
Bir eğitim hata verirse sonrakiler yine çalışır (`|| true`).

**Neden bu dört deney** (README §2 "Değerlendirme bulguları" madde 6): en büyük hatalar yüksek kalorili tabakların eksik tahmini →
(a) çok görevli başlık ve (b) `log1p` hedef; (c) augmentasyonun katkısı (ablation baseline'ı); (d) sınıflandırmada apple pie / tatlı
karışmaları → ViT 384 px. Kalori modellerinde `--patience 10` (ResNet-50 dersi: 7 epoch erken kesebiliyor). Çıta: ConvNeXt-T 41,4 kcal, ViT-B/16 F1 0,951.

In [ ]:
# 1) KURULUM — her oturumda ilk bu hücre çalışır
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "https://github.com/traldo44/kalori-tahmini.git"
GIT_NAME, GIT_EMAIL = "Mert Aydıngüneş", "aydingunesmert@gmail.com"   # ← başkası çalıştırırsa değiştirsin
OWNER = "mert"                                          # ← results.csv'deki owner sütunu

os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GH_TOKEN")
# Token'ı hiçbir dosyaya yazmadan git'e verir; yalnızca bu oturumun hafızasında durur.
subprocess.run(["git", "config", "--global", "credential.helper",
                '!f() { echo username=x-access-token; echo "password=$GH_TOKEN"; }; f'], check=True)

os.chdir("/kaggle/working")
if os.path.isdir("kalori-tahmini"):
    subprocess.run(["git", "-C", "kalori-tahmini", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/kaggle/working/kalori-tahmini")
subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

os.environ["KALORI_DATA"] = "/tmp/kalori-data"   # veri çıktı klasörüne (/kaggle/working) girmesin
os.environ["KALORI_OWNER"] = OWNER
print("Repo hazır →", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2) VERİ — Nutrition5k (~1,4 GB) + Food-101 25 sınıflık alt küme. Aynı oturumda tekrar çalışırsa atlar.
!python scripts/download_nutrition5k.py --out /tmp/kalori-data/nutrition5k
!python scripts/get_food101.py fetch --out /tmp/kalori-data

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok → Settings → Accelerator → GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))

## Kalori ablation'ları (Nutrition5k) — ConvNeXt-T üzerinde, Tablo 2'ye ablation satırları
Çıta: `convnext_tiny_reg` test MAE 41,4 kcal (notebook 11, `--patience 7`).

In [ ]:
# (a) çok görevli başlık: kalori + kütle + yağ + karbonhidrat + protein (~20 dk)
!python -m src.train_reg --backbone convnext_tiny --targets calories mass fat carb protein --patience 10 || true

In [ ]:
# (b) log1p hedef dönüşümü (~20 dk)
!python -m src.train_reg --backbone convnext_tiny --target-transform log1p --patience 10 --name convnext_tiny_reg_log1p || true

In [ ]:
# (c) augmentasyonsuz baseline (~20 dk)  → adı otomatik: convnext_tiny_reg_noaug
!python -m src.train_reg --backbone convnext_tiny --no-aug --patience 10 || true

### Ara kayıt — kalori ablation sonuçları GitHub'a

In [ ]:
!git add results && git commit -q -m "Adım 17: Kaggle kalori ablation'ları (ConvNeXt-T çok görevli, log1p, augmentasyonsuz)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
# yüklenen model dosyası silinir → son hücre aynı dosyayı tekrar yüklemez
!for f in checkpoints/*_reg*.pt; do [ -e "$f" ] && python scripts/share_checkpoint.py upload "$f" && rm "$f"; done

## Sınıflandırma ablation'ı (Food-101) — Tablo 1
Çıta: `vit_b16_cls` test F1 0,951 (224 px).

In [ ]:
# (d) ViT-B/16 384 px (~2,5 saat; 224 px'in ~3 katı)
!python -m src.train_cls --backbone vit_base_patch16_384 --img-size 384 --batch-size 16 --name vit_b16_384_cls || true

## Sonuçları gönder

In [ ]:
!git add results && git commit -q -m "Adım 17: Kaggle sınıflandırma ablation'ı (ViT-B/16 384 px)" && git pull --rebase -q && git push -q && echo "GitHub'a gönderildi ✓"
!for f in checkpoints/*.pt; do [ -e "$f" ] && python scripts/share_checkpoint.py upload "$f" && rm "$f"; done